In [218]:
import os
import json
import random

import numpy as np
import pickle as pk
import pandas as pd
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

In [219]:
from PIL import Image
import torch
import clip

In [220]:
load_dotenv()

True

In [221]:
ASSOCIATION_DIR = os.getenv("ASSOCIATION_DIR")
IMAGE_DIR = os.getenv("IMAGE_DIR")
FINAL_DATASET_IMAGES = os.getenv("FINAL_DATASET_IMAGES")
CRITERIA_EXTRACTION_DIR = os.getenv("CRITERIA_EXTRACTION_DIR")
QUESTIONS_MULTIMODALQA_TRAINING = os.getenv("QUESTIONS_MULTIMODALQA_TRAINING")
selected_questions = pk.load(open("/Users/emanuelemezzi/PycharmProjects/LEGuidance/scripts/selected_questions.pk", "rb"))

In [222]:
sentence_model = SentenceTransformer("all-mpnet-base-v2")

In [223]:
def extract_criteria(model, question): 
    criteria_json = json.load(open(os.path.join(f"{CRITERIA_EXTRACTION_DIR}/{model}/iteration_0", question), "rb"))
    criterias = [criteria_json["topic"]["question_topic"], criteria_json["expected_answer_type"]["expected_answer_type_specific"], 
                 criteria_json["target"]["type"], criteria_json["asked_property"]]
        
    constraints = [constraint["evidence"] for constraint in criteria_json["constraints"]]
    time_constraints = [time_constraint["label"] for time_constraint in criteria_json["time_constraints"]]
    criterias.extend(constraints)
    criterias.extend(time_constraints)
    
    return criterias

In [224]:
def extract_metadata(question): 
    association_json = json.load(open(os.path.join(ASSOCIATION_DIR, question), "rb"))
        
    # Extract metadata images
    data_images = [{"metadata": image["title"], "data": image["path"]} for image in association_json["image_set"]]
    
    # Extract metadata paragraphs
    data_text = [{"metadata": text["title"], "data": text["text"]} for text in association_json["text_set"]]
    
    # Extract metadata table
    metadata_table = []
    for table in association_json["table_set"]: 
        table_code = table["json"]
        table_json = json.load(open(os.path.join("/Users/emanuelemezzi/Desktop/datasetNIPS/multimodalqa_files/tables", table_code), "rb"))
        metadata_table.extend([table_json["title"], table_json["table"]["table_name"]])
        metadata_table.extend([header["column_name"] for header in table_json["table"]["header"]])
    
    metadata_table = [el for el in metadata_table if el != '']
        
    # print("Metadata images: ", metadata_images)
    # print("Metadata text: ", metadata_text)
    # print("Metadata table: ", metadata_table)
        
    return data_images, data_text, metadata_table, association_json

In [225]:
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, preprocess = clip.load("ViT-B/32", device=device)

In [226]:
def measure_similarity(criterias, data_images, data_text, metadata_table): 
    
    metadata_sim = {}
    
    # Measure similarity with images
    if data_images:  
        metadata_images = [image["metadata"] for image in data_images]
        data_images = [image["path"] for image in data_images]
        
        text_tokens = clip.tokenize(metadata_images).to(device)
        images = torch.stack([preprocess(Image.open(os.path.join(IMAGE_DIR, path))) for path in data_images]).to(device)
        
        criteria_tokens = clip.tokenize(criterias).to(device)
        
        with torch.no_grad(): 
            metadata_embeddings = clip_model.encode_text(text_tokens)
            image_embeddings = clip_model.encode_image(images)
            criteria_embeddings = clip_model.encode_text(criteria_tokens)
            
        metadata_embeddings /= metadata_embeddings.norm(dim=-1, keepdim=True)
        image_embeddings /= image_embeddings.norm(dim=-1, keepdim=True)
        criteria_embeddings /= criteria_embeddings.norm(dim=-1, keepdim=True)
        
        joint_embeddings = (metadata_embeddings + image_embeddings) / 2
        joint_embeddings = joint_embeddings / joint_embeddings.norm(dim=-1, keepdim=True)
        
        similarity_images = criteria_embeddings @ joint_embeddings.T
        similarity_images = similarity_images.softmax(dim=-1).cpu().numpy()
        
        df_sim_images = pd.DataFrame(
            similarity_images,
            index=criterias, 
            columns=metadata_images
        )
        
        """
        # metadata_images_embeddings = clip_model.encode_text(metadata_images)
        # metadata_images_embeddings = model.encode(metadata_images)
        # similarity_images = model.similarity(criteria_embeddings, metadata_images_embeddings)
        df_sim_images = pd.DataFrame(
            similarity_images.cpu().numpy(),   # convert tensor → numpy
            index=criterias,                   # rows = criteria sentences
            columns=metadata_images            # columns = metadata sentences
        )
        """
        
        df_sim_images["max_similarity"] = df_sim_images.max(axis=1)
        metadata_sim["image"] = df_sim_images["max_similarity"].mean()
    
    # Measure similarity with text 
    if data_text: 
        criteria_embeddings = model.encode(criterias)
        metadata_text_embeddings = model.encode(data_text)
        similarity_text = model.similarity(criteria_embeddings, metadata_text_embeddings)
        df_sim_text = pd.DataFrame(
            similarity_text.cpu().numpy(), 
            index=criterias, 
            columns=data_text
        )
        df_sim_text["max_similarity"] = df_sim_text.max(axis=1)
        metadata_sim["text"] = df_sim_text["max_similarity"].mean()
    
    # Measure similarity with table
    if metadata_table: 
        metadata_table_embeddings = model.encode(metadata_table)
        similarity_table = model.similarity(criteria_embeddings, metadata_table_embeddings)
        df_sim_table = pd.DataFrame(
            similarity_table.cpu().numpy(), 
            index=criterias, 
            columns=metadata_table
        )
        df_sim_table["max_similarity"] = df_sim_table.max(axis=1)
        metadata_sim["table"] = df_sim_table["max_similarity"].mean()
    
    extracted_modalities = [k for k, v in sorted(metadata_sim.items(), key=lambda x: x[1], reverse=True)]
    
    return extracted_modalities

In [288]:
def measure_similarity(question, criterias, data_images, data_text, metadata_table): 
    
    metadata_sim = {}
    
    original_question = [json.load(open(os.path.join(QUESTIONS_MULTIMODALQA_TRAINING, question), "rb"))["question"]][:]
    
    # Measure similarity with images
    if data_images:  
        metadata_images = [image["metadata"] for image in data_images]
        path_images = [image["data"] for image in data_images]
        
        text_tokens = clip.tokenize(metadata_images).to(device)
        images = torch.stack([preprocess(Image.open(os.path.join(FINAL_DATASET_IMAGES, path)).convert("RGB")) for path in path_images]).to(device)
        criteria_tokens = clip.tokenize(original_question).to(device)
        
        with torch.no_grad(): 
            criteria_embeddings = clip_model.encode_text(criteria_tokens)
            metadata_embeddings = clip_model.encode_text(text_tokens)
            image_embeddings = clip_model.encode_image(images)
            
        metadata_embeddings /= metadata_embeddings.norm(dim=-1, keepdim=True)
        image_embeddings /= image_embeddings.norm(dim=-1, keepdim=True)
        criteria_embeddings /= criteria_embeddings.norm(dim=-1, keepdim=True)
        
        joint_embeddings = (metadata_embeddings + image_embeddings) / 2
        joint_embeddings = joint_embeddings / joint_embeddings.norm(dim=-1, keepdim=True)
        
        similarity_images = criteria_embeddings @ joint_embeddings.T
        similarity_images = similarity_images.softmax(dim=-1).cpu().numpy()
        
        df_sim_images = pd.DataFrame(
            similarity_images,
            index=original_question, 
            columns=metadata_images
        )
       
        df_sim_images["max_similarity"] = df_sim_images.max(axis=1)
        metadata_sim["image"] = df_sim_images["max_similarity"].mean()
    
    # Measure similarity with text 
    if data_text: 
        metadata_text = [data["metadata"] for data in data_text]
        texts = [data["data"] for data in data_text]
        
        data = [f"Title: {t}. Content: {c}" for t, c in zip(metadata_text, texts)]
        
        criteria_embeddings = sentence_model.encode(original_question)
        text_embeddings = sentence_model.encode(data)
        
        similarity_text = sentence_model.similarity(criteria_embeddings, text_embeddings)
        df_sim_text = pd.DataFrame(
            similarity_text.cpu().numpy(), 
            index=original_question, 
            columns=data
        )
        
        df_sim_text["max_similarity"] = df_sim_text.max(axis=1)
        metadata_sim["text"] = df_sim_text["max_similarity"].mean()
    
    # Measure similarity with table
    if metadata_table: 
        criteria_embeddings = sentence_model.encode(original_question)
        metadata_table_embeddings = sentence_model.encode(metadata_table)
        
        similarity_table = sentence_model.similarity(criteria_embeddings, metadata_table_embeddings)
        df_sim_table = pd.DataFrame(
            similarity_table.cpu().numpy(), 
            index=original_question, 
            columns=metadata_table
        )
        
        df_sim_table["max_similarity"] = df_sim_table.max(axis=1)
        metadata_sim["table"] = df_sim_table["max_similarity"].mean()
    
    extracted_modalities = [k for k, v in sorted(metadata_sim.items(), key=lambda x: x[1], reverse=True)]
    
    return extracted_modalities

In [289]:
def check_accuracy(modalities_extracted, modalities_gt, order):
    k = len(modalities_gt)

    # Consider only top-k predictions
    top_k = modalities_extracted[:k]

    if order:
        # Order matters → exact position comparison
        correct = sum(1 for pred, gt in zip(top_k, modalities_gt) if pred == gt)
        return correct / k  # returns 1, 0.5, or 0
    else:
        # Order does not matter → set intersection
        intersection = set(top_k).intersection(set(modalities_gt))
        return len(intersection) / k  

In [294]:
import random

def main(model, selected_questions):
    results = {}
    random.shuffle(selected_questions)
    for i, question in enumerate(selected_questions[:1000]): 
        try:
            print("The question is: ", i, question)
            """Extract the criterias"""
            criterias = extract_criteria(model, question)
            
            metadata_images, metadata_text, metadata_table, association_json = extract_metadata(question)
            
            modalities_extracted = measure_similarity(question, criterias, metadata_images, metadata_text, metadata_table)
            print("Modalities extracted: ", modalities_extracted)
            modalities_gt = json.load(open(os.path.join(QUESTIONS_MULTIMODALQA_TRAINING, question), "rb"))["metadata"]["modalities"]
            print("Gt for this question: ", modalities_gt)
            
            if tuple(modalities_gt) not in results: 
                results[tuple(modalities_gt)] = {}
                results[tuple(modalities_gt)]["ordered"], results[tuple(modalities_gt)]["unordered"] = {}, {}
       
            results[tuple(modalities_gt)]["ordered"][question] = check_accuracy(modalities_extracted, modalities_gt, True)
            results[tuple(modalities_gt)]["unordered"][question] = check_accuracy(modalities_extracted, modalities_gt, False)
        
        except Exception as e:
            print(f"Skipping question {question} due to error: {e}")
            continue
    
    return results

In [295]:
results = main("gpt-5.2", selected_questions)

The question is:  0 question_20401.json
Modalities extracted:  ['text', 'table', 'image']
Gt for this question:  ['table', 'text']
The question is:  1 question_18059.json
Modalities extracted:  ['table', 'text', 'image']
Gt for this question:  ['table']
The question is:  2 question_8618.json
Modalities extracted:  ['table', 'text', 'image']
Gt for this question:  ['image', 'text']
The question is:  3 question_11346.json
Modalities extracted:  ['text', 'table', 'image']
Gt for this question:  ['table', 'image']
The question is:  4 question_9417.json
Modalities extracted:  ['table', 'text', 'image']
Gt for this question:  ['table']
The question is:  5 question_21688.json
Modalities extracted:  ['text', 'image', 'table']
Gt for this question:  ['text']
The question is:  6 question_19637.json
Modalities extracted:  ['table', 'text', 'image']
Gt for this question:  ['image', 'text']
The question is:  7 question_1602.json
Modalities extracted:  ['table', 'text', 'image']
Gt for this question

/Users/emanuelemezzi/PycharmProjects/LogicalEntropyGuidance/lib/python3.10/site-packages/PIL/Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Modalities extracted:  ['table', 'text', 'image']
Gt for this question:  ['image', 'table']
The question is:  26 question_179.json
Modalities extracted:  ['table', 'text', 'image']
Gt for this question:  ['image', 'table']
The question is:  27 question_10604.json
Modalities extracted:  ['text', 'table', 'image']
Gt for this question:  ['text', 'table']
The question is:  28 question_5809.json
Modalities extracted:  ['table', 'text', 'image']
Gt for this question:  ['table', 'table']
The question is:  29 question_15064.json
Modalities extracted:  ['table', 'text', 'image']
Gt for this question:  ['table', 'text']
The question is:  30 question_21325.json
Modalities extracted:  ['table', 'text', 'image']
Gt for this question:  ['image', 'table']
The question is:  31 question_21593.json
Modalities extracted:  ['table', 'text', 'image']
Gt for this question:  ['table', 'table']
The question is:  32 question_14276.json
Modalities extracted:  ['table', 'text', 'image']
Gt for this question:  [

/Users/emanuelemezzi/PycharmProjects/LogicalEntropyGuidance/lib/python3.10/site-packages/PIL/Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Modalities extracted:  ['text', 'table', 'image']
Gt for this question:  ['text', 'table']
The question is:  70 question_9660.json
Modalities extracted:  ['text', 'image', 'table']
Gt for this question:  ['text']
The question is:  71 question_13095.json
Modalities extracted:  ['text', 'table', 'image']
Gt for this question:  ['table', 'text']
The question is:  72 question_7764.json
Modalities extracted:  ['table', 'text', 'image']
Gt for this question:  ['image', 'table']
The question is:  73 question_12858.json
Modalities extracted:  ['text', 'image', 'table']
Gt for this question:  ['text']
The question is:  74 question_17821.json
Modalities extracted:  ['text', 'table', 'image']
Gt for this question:  ['text', 'table']
The question is:  75 question_7574.json
Modalities extracted:  ['table', 'text', 'image']
Gt for this question:  ['image', 'text']
The question is:  76 question_17164.json
Modalities extracted:  ['text', 'table', 'image']
Gt for this question:  ['image']
The question 

...

Modalities extracted:  ['table', 'text', 'image']
Gt for this question:  ['image', 'table']
The question is:  917 question_6750.json
Modalities extracted:  ['text', 'table', 'image']
Gt for this question:  ['image']
The question is:  918 question_20086.json
Modalities extracted:  ['table', 'text', 'image']
Gt for this question:  ['text', 'table']
The question is:  919 question_22337.json
Modalities extracted:  ['text', 'table', 'image']
Gt for this question:  ['table']
The question is:  920 question_16755.json
Modalities extracted:  ['text', 'table', 'image']
Gt for this question:  ['image', 'text']
The question is:  921 question_17993.json
Modalities extracted:  ['text', 'table', 'image']
Gt for this question:  ['image', 'table']
The question is:  922 question_5100.json
Modalities extracted:  ['text', 'table', 'image']
Gt for this question:  ['image', 'text']
The question is:  923 question_10324.json
Modalities extracted:  ['text', 'table', 'image']
Gt for this question:  ['text']
The

/Users/emanuelemezzi/PycharmProjects/LogicalEntropyGuidance/lib/python3.10/site-packages/PIL/Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Modalities extracted:  ['text', 'table', 'image']
Gt for this question:  ['text']
The question is:  986 question_2908.json
Modalities extracted:  ['text', 'table', 'image']
Gt for this question:  ['table', 'text']
The question is:  987 question_11262.json
Modalities extracted:  ['text', 'table', 'image']
Gt for this question:  ['text']
The question is:  988 question_22158.json
Modalities extracted:  ['text', 'table', 'image']
Gt for this question:  ['text']
The question is:  989 question_12730.json
Modalities extracted:  ['text', 'table', 'image']
Gt for this question:  ['image']
The question is:  990 question_1324.json
Modalities extracted:  ['text', 'table', 'image']
Gt for this question:  ['text']
The question is:  991 question_17040.json


/Users/emanuelemezzi/PycharmProjects/LogicalEntropyGuidance/lib/python3.10/site-packages/PIL/Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Modalities extracted:  ['text', 'table', 'image']
Gt for this question:  ['image', 'table']
The question is:  992 question_6448.json
Modalities extracted:  ['text', 'table', 'image']
Gt for this question:  ['text']
The question is:  993 question_10726.json
Modalities extracted:  ['text', 'table', 'image']
Gt for this question:  ['table']
The question is:  994 question_23156.json
Modalities extracted:  ['text', 'image', 'table']
Gt for this question:  ['text']
The question is:  995 question_17349.json
Modalities extracted:  ['text', 'table', 'image']
Gt for this question:  ['image', 'table']
The question is:  996 question_16139.json
Modalities extracted:  ['table', 'text', 'image']
Gt for this question:  ['table', 'text']
The question is:  997 question_3834.json
Modalities extracted:  ['text', 'table', 'image']
Gt for this question:  ['table', 'text']
The question is:  998 question_23438.json
Modalities extracted:  ['text', 'table', 'image']
Gt for this question:  ['table', 'text']
The 

In [296]:
print(results)

{('table', 'text'): {'ordered': {'question_20401.json': 0.0, 'question_3470.json': 0.0, 'question_12619.json': 0.0, 'question_12515.json': 1.0, 'question_15064.json': 1.0, 'question_9184.json': 1.0, 'question_19664.json': 1.0, 'question_13088.json': 1.0, 'question_13095.json': 0.0, 'question_8533.json': 0.0, 'question_15865.json': 0.0, 'question_14438.json': 0.0, 'question_6517.json': 1.0, 'question_10439.json': 0.0, 'question_19307.json': 0.0, 'question_13935.json': 0.0, 'question_19837.json': 1.0, 'question_19777.json': 0.0, 'question_12505.json': 0.0, 'question_4014.json': 1.0, 'question_10125.json': 1.0, 'question_15505.json': 1.0, 'question_13262.json': 0.0, 'question_2276.json': 0.0, 'question_14161.json': 1.0, 'question_13742.json': 0.0, 'question_11998.json': 0.0, 'question_13716.json': 1.0, 'question_6284.json': 0.0, 'question_9033.json': 1.0, 'question_13743.json': 0.0, 'question_5051.json': 0.0, 'question_17359.json': 1.0, 'question_19450.json': 0.0, 'question_14648.json': 0

In [297]:
rows = []

for modality in results.keys():
    acc_ordered = np.mean(list(results[modality]["ordered"].values()))
    acc_unordered = np.mean(list(results[modality]["unordered"].values()))
    
    # Append row as a dictionary
    rows.append({
        "Modality": ', '.join(modality),  # convert tuple to string
        "Accuracy_Ordered": acc_ordered,
        "Accuracy_Unordered": acc_unordered
    })
    
    # Optional: print while executing
    print(f"Modality: {modality}")
    print(f"Accuracy ordered: {acc_ordered}")
    print(f"Accuracy unordered: {acc_unordered}")

# Create DataFrame
df = pd.DataFrame(rows)

Modality: ('table', 'text')
Accuracy ordered: 0.4142857142857143
Accuracy unordered: 1.0
Modality: ('table',)
Accuracy ordered: 0.464
Accuracy unordered: 0.464
Modality: ('image', 'text')
Accuracy ordered: 0.21904761904761905
Accuracy unordered: 0.5
Modality: ('table', 'image')
Accuracy ordered: 0.26744186046511625
Accuracy unordered: 0.5
Modality: ('text',)
Accuracy ordered: 0.9937106918238994
Accuracy unordered: 0.9937106918238994
Modality: ('text', 'table')
Accuracy ordered: 0.5324675324675324
Accuracy unordered: 1.0
Modality: ('image',)
Accuracy ordered: 0.006622516556291391
Accuracy unordered: 0.006622516556291391
Modality: ('image', 'table')
Accuracy ordered: 0.24479166666666666
Accuracy unordered: 0.5
Modality: ('table', 'table')
Accuracy ordered: 0.4849624060150376
Accuracy unordered: 0.4849624060150376
Modality: ('text', 'image')
Accuracy ordered: 0.4722222222222222
Accuracy unordered: 0.5


In [298]:
df

,Modality,Accuracy_Ordered,Accuracy_Unordered
0,"table, text",0.414286,1.000000
1,table,0.464000,0.464000
2,"image, text",0.219048,0.500000
3,"table, image",0.267442,0.500000
4,text,0.993711,0.993711
5,"text, table",0.532468,1.000000
6,image,0.006623,0.006623
7,"image, table",0.244792,0.500000
8,"table, table",0.484962,0.484962
9,"text, image",0.472222,0.500000
